In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import sys
import os
import logging
import time
import json
from datetime import datetime
import re
import subprocess


In [ ]:
spark = SparkSession.builder \
    .appName("DataProcessingApp") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.cores", "2") \
    .getOrCreate()

spark

* **header=True:** Treats the first row as column names.
* **inferSchema=True:** Automatically detects data types.
* **Efficient for loading structured data into a distributed format.**

In [ ]:
dfcsv = spark.read.csv("/kaggle/input/human-resources-data-set/HRDataset_v14.csv", header=True, inferSchema=True)
dfcsv.show()

**Printing DataFrame Schema** 
_To view the structure (columns and data types):_

In [ ]:
dfcsv.printSchema()

**Common tasks:** 
* Row count
* Group and aggregate:

In [ ]:
from pyspark.sql import functions as F

totalcount = dfcsv.count()

dfcsv.groupBy("GenderID").agg({"Salary": "sum", "Salary": "avg"}).show()


dfcsv.groupBy("GenderID").agg(
    F.sum("Salary").alias("Salary_Sum"),
    F.avg("Salary").alias("Salary_Avg")
).show()

**Key DataFrame methods:**
| *Function* |*Use Case* | *Equivalent in SQL* |
| ---- |---- | ---- |
| select() | Choose specific columns | SELECT |
| filter() | Filter rows by condition | WHERE |
| groupBy() | Group rows by one or more columns | GROUP BY |
| agg() | Aggregate with a function like sum, avg | AGGREGATE |

**SQL:** _Select Employee_Name, HispanicLatino from csv_file where State = 'MA'_

In [ ]:
dfcsv.filter(dfcsv.State == "MA").select("Employee_Name", "HispanicLatino").show() 

*To get employees whose salary is 40% above the average salary, it need to:*
1. 	Calculate the total salary across all employees.
2. 	Compute the threshold (i.e., total salary × 1.4).
3. 	Filter employees whose salary exceeds that threshold.

In [ ]:
from pyspark.sql import functions as F

# Step 1: Calculate average salary
avg_sal = dfcsv.agg(F.avg("Salary").alias("Avg_Salary")).collect()[0]["Avg_Salary"]

print(f"Average Salary: {avg_sal}")

# Step 2: Compute 40% above total salary
threshold = avg_sal * 1.4

# Step 3: Filter employees whose salary exceeds the threshold
dfcsv.filter(dfcsv.Salary > threshold) \
     .withColumn("Avg_Salary", F.lit(avg_sal)) \
     .select("Employee_Name", "Salary", "Avg_Salary") \
     .show()


*Count the total number of rows*

In [ ]:
row_count = dfcsv.count()
print(f"Total rows: {row_count}")

*Applying more than one filter*

In [ ]:
# Option 1: Chaining .filter() Calls
dfcsv.filter((col("GenderID") == 1) & (col("Salary") > 60000)).show(5)

# Option 2: Single .filter() with Logical Operators
# Use  for & AND,  | for OR, and  ~ for NOT.
dfcsv.filter((col("State") == "MA") & (col("Salary") > 60000)).show(5)

# Option 3: Using .where() Method
dfcsv.where((col("MarriedID") == 0) & (col("Salary") > 70000)).show(5)

#Filter + Select
dfcsv.filter((col("GenderID") == 1) & (col("Salary") > 60000)) \
     .select("Employee_Name", "Salary", "GenderID") \
     .show(5)

>#### Schema Inference and Manual Schema Definition:
>>Spark can automatically infer schemas, but it might misinterpret data types, particularly with complex or ambiguous data. Manually defining a schema can ensure accurate data handling.

>#### DataTypes in PySpark DataFrames 
>>To manually configure a schema, we define the datatype using the StructField function, calling the appropriate datatype method. PySpark DataFrames support various data types, similar to SQL and Pandas.

In [ ]:
# Show Datatypes
# Option 1: Use .printSchema() for a Tree View

#dfcsv.printSchema()

# Option 2: Use .dtypes for a List of Tuples
dfcsv.dtypes

# Option 3: Use .schema for a Detailed Schema Object
dfcsv.schema

# OR Convert to Pandas for Quick Profiling
# If you want to do a quick audit or export to Pandas for profiling, you can convert a sample to Pandas:
dfcsv.limit(100).toPandas().info()


### Display list of Columns

In [ ]:
# Option 1: Use  to List All Column Names
dfcsv.columns
# Option 2: Use .select("*").show() with truncate=False
dfcsv.select("*").show(truncate=False, n=5)
# Option 3: Convert to Pandas (for Small Datasets)
dfcsv.limit(5).toPandas().head()
# Option 4: Use .take() for a List of Rows
dfcsv.take(5)
# Print Schema for Column Names + Types
dfcsv.printSchema()

#### Finding NULL or NOT NULL
>
>> In PySpark, filtering for NULL values is just as intuitive as SQL’s WHERE column IS NULL; use the .isNull() method on the column.


In [ ]:
# Filter Rows Where a Column Is NULL
# SQL : SELECT * FROM dfcsv WHERE DateofTermination IS NULL;
dfcsv.filter(dfcsv.DateofTermination.isNull()).show(5)

# Filter Rows Where a Column Is NOT NULL
# SQL : SELECT * FROM dfcsv WHERE DateofTermination IS NOT NULL;
dfcsv.filter(dfcsv.DateofTermination.isNotNull()).show(5)

# Combine with Other Filters
# WHERE DateofTermination IS NULL AND State = 'MA'.
dfcsv.filter((dfcsv.DateofTermination.isNull()) & (dfcsv.State == "MA")).show(7)


### Handling Missing Data
**Handling null values is crucial in data analysis as missing data can lead to skewed results or errors during processing. PySpark provides two primary methods for managing missing values:**
>
>> #### Dropping Null Values:
>>
>>> * The .na.drop() method can be used to drop rows that contain null values. 
>>> * This can be done across the entire DataFrame or for specific columns. 
>>> * While this method simplifies the dataset, it might significantly reduce the dataset size if null values are common, which could lead to loss of important data.